**DANIEL YU & JORDAN WANG**

Spring 2026

CS 443: Bio-inspired Machine Learning

Project 2: Predictive Coding

#### Extension 4: Hyperparameter Exploration with ConvPCN

For this extension we try a small set of ConvPCN hyperparameter changes and compare how they affect CIFAR-10 validation accuracy. The goal is not to brute force search everything, but to test a few reasonable ideas and see what actually helps.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

plt.style.use(['seaborn-v0_8-colorblind', 'seaborn-v0_8-darkgrid'])
plt.rcParams.update({'font.size': 16})
np.set_printoptions(suppress=True, precision=3)

%load_ext autoreload
%autoreload 2

### Plan

We keep the architecture mostly fixed and compare a few training settings:
1. Baseline settings from Task 10.
2. Lower learning rate.
3. Larger batch size.
4. More dropout.

This keeps the experiment focused and makes the results easier to explain.

In [2]:
from image_datasets import get_dataset, train_val_split
from conv_pcn import ConvPCN6Mini

In [3]:
# Load CIFAR-10 in image form
x_train_full, y_train_full, x_test, y_test = get_dataset('cifar10', norm_method='global', flatten=False)
x_train, y_train, x_val, y_val = train_val_split(x_train_full, y_train_full, prop_val=0.1)
print(f'Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}')

/Users/jordanwang/Desktop/CS443/.venv/lib/python3.13/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Dataset: cifar10
x_train: (50000, 32, 32, 3) <dtype: 'float32'>
y_train: (50000,) <dtype: 'uint8'>
x_test:  (10000, 32, 32, 3) <dtype: 'float32'>
y_test:  (10000,) <dtype: 'uint8'>
Train: (45000, 32, 32, 3), Val: (5000, 32, 32, 3), Test: (10000, 32, 32, 3)


### Hyperparameter Configs

All runs use He initialization and GroupNorm so the comparison is about training hyperparameters, not a weaker network setup.

In [4]:
configs = [
    {'label': 'A. baseline',      'lr': 1e-4, 'batch_size': 256, 'dropout_rate': 0.20, 'num_steps': 5},
    {'label': 'B. lower lr',      'lr': 5e-5, 'batch_size': 256, 'dropout_rate': 0.20, 'num_steps': 5},
    {'label': 'C. larger batch',  'lr': 1e-4, 'batch_size': 512, 'dropout_rate': 0.20, 'num_steps': 5},
    {'label': 'D. more dropout',  'lr': 1e-4, 'batch_size': 256, 'dropout_rate': 0.35, 'num_steps': 5},
]
configs

[{'label': 'A. baseline',
  'lr': 0.0001,
  'batch_size': 256,
  'dropout_rate': 0.2,
  'num_steps': 5},
 {'label': 'B. lower lr',
  'lr': 5e-05,
  'batch_size': 256,
  'dropout_rate': 0.2,
  'num_steps': 5},
 {'label': 'C. larger batch',
  'lr': 0.0001,
  'batch_size': 512,
  'dropout_rate': 0.2,
  'num_steps': 5},
 {'label': 'D. more dropout',
  'lr': 0.0001,
  'batch_size': 256,
  'dropout_rate': 0.35,
  'num_steps': 5}]

### Train Runs

If this is running on a slow machine or cloud notebook, turn `quick_mode` on first to check everything is wired correctly.

In [5]:
results = []

quick_mode = False

if quick_mode:
    x_train_run, y_train_run = x_train[:12000], y_train[:12000]
    x_val_run, y_val_run = x_val[:2500], y_val[:2500]
    max_epochs = 8
else:
    x_train_run, y_train_run = x_train, y_train
    x_val_run, y_val_run = x_val, y_val
    max_epochs = 20

for cfg in configs:
    print('\n' + '='*64)
    print(f"Training {cfg['label']}")
    print('='*64)

    tf.random.set_seed(42)
    net = ConvPCN6Mini(
        input_feats_shape=(32, 32, 3),
        C=10,
        dropout_rate=cfg['dropout_rate'],
        num_steps=cfg['num_steps'],
        wt_init='he',
        do_group_norm=True
    )
    net.compile(loss='cross_entropy', lr=cfg['lr'], print_summary=False)

    train_loss, val_loss, val_acc, epochs_used = net.fit(
        x_train_run, y_train_run,
        x_val=x_val_run, y_val=y_val_run,
        batch_size=cfg['batch_size'],
        max_epochs=max_epochs,
        patience=7,
        lr_patience=3,
        lr_max_decays=3,
        verbose=True,
        print_every=1
    )

    results.append({
        'label': cfg['label'],
        'cfg': cfg,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'epochs': epochs_used,
        'best_val_acc': float(np.max(val_acc)),
        'best_epoch': int(np.argmax(val_acc) + 1),
        'net': net
    })

    print(f"Best val acc: {np.max(val_acc):.4f} @ epoch {np.argmax(val_acc)+1}")
    print(f"Final val acc: {val_acc[-1]:.4f}")


Training A. baseline


KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for r in results:
    epochs = np.arange(1, len(r['train_loss']) + 1)
    axes[0].plot(epochs, r['train_loss'], linewidth=2, label=r['label'])
    axes[1].plot(epochs, r['val_loss'], linewidth=2, label=r['label'])
    axes[2].plot(epochs, r['val_acc'], linewidth=2, label=r['label'])

titles = ['Training Loss', 'Validation Loss', 'Validation Accuracy']
ylabs = ['Loss', 'Loss', 'Accuracy']
for ax, t, y in zip(axes, titles, ylabs):
    ax.set_title(t)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(y)
    ax.legend(fontsize=10)

plt.suptitle('Extension 4: ConvPCN Hyperparameter Sweep', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print('Validation summary:')
print('-'*70)
for r in results:
    cfg = r['cfg']
    print(f"{r['label']}: lr={cfg['lr']}, bs={cfg['batch_size']}, drop={cfg['dropout_rate']} | best={r['best_val_acc']:.4f} (epoch {r['best_epoch']})")

In [ ]:
best_idx = int(np.argmax([r['best_val_acc'] for r in results]))
best = results[best_idx]
test_acc, test_loss = best['net'].evaluate(x_test, y_test)
print(f"Best config: {best['label']}")
print(f"Test accuracy: {float(test_acc):.4f}")
print(f"Test loss: {float(test_loss):.4f}")

### Results and Interpretation

This experiment is mostly about relative comparisons, not one absolute score. The main things to look at are:
- which setting reaches the best validation accuracy,
- whether training looks stable or noisy,
- whether the validation curve starts to flatten or drop off early.

If a config has a small gain but takes much longer or becomes less stable, it is not necessarily the best tradeoff. For this project, a simple improvement with clear behavior is better than a complicated sweep with no story behind it.

### Extension 4 Conclusion

We completed Extension 4 by comparing a small set of ConvPCN hyperparameter settings on CIFAR-10 and using validation behavior to decide which configuration performed best.